In [1]:
import pandas as pd
import requests

In [2]:
# URL oficial de datos.gov.co
URL_DIVIPOLA = 'https://www.datos.gov.co/api/views/vafm-j2df/rows.csv?accessType=DOWNLOAD'

print("Descargando DIVIPOLA...")
df_div = pd.read_csv(URL_DIVIPOLA, low_memory=False)
print(f"Shape original: {df_div.shape}")
print(f"Columnas originales: {df_div.columns.tolist()}")

Descargando DIVIPOLA...
Shape original: (1121, 8)
Columnas originales: ['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO', 'TIPO', 'LATITUD', 'LONGITUD', 'Geo Municipio']


In [3]:
# Limpiar y quedarnos con lo necesario
df_div = df_div[['COD_DPTO','NOM_DPTO','COD_MPIO','NOM_MPIO','LATITUD','LONGITUD']].copy()

# Tipos
df_div['LATITUD']  = pd.to_numeric(df_div['LATITUD'],  errors='coerce')
df_div['LONGITUD'] = pd.to_numeric(df_div['LONGITUD'], errors='coerce')
df_div = df_div.dropna(subset=['LATITUD','LONGITUD'])

# Código municipio canónico (5 dígitos con zfill)
df_div['cod_municipio'] = df_div['COD_MPIO'].astype(int).astype(str).str.zfill(5)
df_div['cod_departamento'] = df_div['COD_DPTO'].astype(int).astype(str).str.zfill(2)

# Renombrar a snake_case (consistencia con el resto del lakehouse)
df_div = df_div.rename(columns={
    'NOM_DPTO': 'nom_departamento',
    'NOM_MPIO': 'nom_municipio',
    'LATITUD': 'latitud',
    'LONGITUD': 'longitud'
})

# Columnas finales
df_div = df_div[['cod_municipio', 'nom_municipio', 'cod_departamento', 'nom_departamento', 'latitud', 'longitud']]

# Validaciones
print(f"\nMunicipios únicos: {df_div['cod_municipio'].nunique()}")
print(f"Departamentos únicos: {df_div['cod_departamento'].nunique()}")
print(f"\nNulos:\n{df_div.isna().sum()}")
print(f"\nRango lat: [{df_div['latitud'].min():.2f}, {df_div['latitud'].max():.2f}]")
print(f"Rango lon: [{df_div['longitud'].min():.2f}, {df_div['longitud'].max():.2f}]")
print(f"\nMuestra:")
print(df_div.head())


Municipios únicos: 1121
Departamentos únicos: 33

Nulos:
cod_municipio       0
nom_municipio       0
cod_departamento    0
nom_departamento    0
latitud             0
longitud            0
dtype: int64

Rango lat: [-3.63, 13.35]
Rango lon: [-81.72, -67.00]

Muestra:
  cod_municipio nom_municipio cod_departamento nom_departamento   latitud  \
0         05001      MEDELLÍN               05        ANTIOQUIA  6.257590   
1         05002     ABEJORRAL               05        ANTIOQUIA  5.803728   
2         05004      ABRIAQUÍ               05        ANTIOQUIA  6.627569   
3         05021    ALEJANDRÍA               05        ANTIOQUIA  6.365534   
4         05030         AMAGÁ               05        ANTIOQUIA  6.032922   

    longitud  
0 -75.611031  
1 -75.438474  
2 -76.085978  
3 -75.090597  
4 -75.708003  


In [4]:
# Guardar
df_div.to_parquet("divipola.parquet", compression='gzip')
print("\n✅ divipola.parquet generado")


✅ divipola.parquet generado
